# Masiez — Optional External Drone Link (LiveKit)

Autonomy runs **in the browser** by default. Use this notebook only when models or controllers must run **outside** the browser: it connects to the simulated FPV drone over WebRTC, streams live frames and telemetry, teleoperates the aircraft, invokes robot RPCs, and records synchronized datasets for Kaggle training.

Requires the optional LiveKit stack to be running (`--profile livekit`).

In [ ]:
import asyncio
import io
import json
import os
import time
from pathlib import Path

import numpy as np
from PIL import Image
from livekit import api, rtc
from IPython.display import display, Image as IPyImage, clear_output

# LiveKit SFU Configuration
LIVEKIT_URL = os.environ.get("LIVEKIT_URL", "ws://localhost:7880")
LIVEKIT_API_KEY = os.environ.get("LIVEKIT_API_KEY", "devkey")
LIVEKIT_API_SECRET = os.environ.get("LIVEKIT_API_SECRET", "devsecret_devsecret_devsecret_1234")
LIVEKIT_ROOM = os.environ.get("LIVEKIT_ROOM", "drone")
ROBOT_IDENTITY = "drone-sim"
CONTROLLER_IDENTITY = "jupyter-controller"

def mint_controller_token(identity: str = CONTROLLER_IDENTITY) -> str:
    """Mint a LiveKit JWT token for this Jupyter notebook teleoperator."""
    return (
        api.AccessToken(LIVEKIT_API_KEY, LIVEKIT_API_SECRET)
        .with_identity(identity)
        .with_grants(
            api.VideoGrants(
                room_join=True,
                room=LIVEKIT_ROOM,
                can_publish=False,
                can_subscribe=True,
                can_publish_data=True,
            )
        )
        .to_jwt()
    )

print("LiveKit Robotics environment configured for room:", LIVEKIT_ROOM)

## 1. LiveKit Robot Client (WebRTC Video, DataChannels & RPC)
Subscribes to `drone-fpv` video track, listens to telemetry and kamikaze strike events, and provides teleoperation helpers.

In [ ]:
class LiveKitDroneRobotClient:
    def __init__(self):
        self.room = rtc.Room()
        self.latest_frame: np.ndarray | None = None
        self.latest_telemetry: dict | None = None
        self.latest_mission: dict | None = None
        self.strike_events: list[dict] = []
        self._video_task: asyncio.Task | None = None

        @self.room.on("track_subscribed")
        def _on_track(track: rtc.Track, pub: rtc.RemoteTrackPublication, participant: rtc.RemoteParticipant):
            if track.kind == rtc.TrackKind.KIND_VIDEO:
                print(f"[LiveKit] Subscribed to camera track '{pub.name}' from '{participant.identity}'")
                self._video_task = asyncio.create_task(self._consume_video(track))

        @self.room.on("data_received")
        def _on_data(data_packet: rtc.DataPacket):
            try:
                text = bytes(data_packet.data).decode("utf-8")
                msg = json.loads(text)
                if msg.get("type") == "drone:telemetry":
                    self.latest_telemetry = msg.get("telemetry")
                    if "mission" in msg:
                        self.latest_mission = msg.get("mission")
                elif msg.get("type") == "drone:kamikaze_strike":
                    self.strike_events.append(msg)
                    self.latest_mission = msg.get("mission")
                    print(f"💥 [STRIKE EVENT] Reason: {msg.get('reason')} | Targets: {msg.get('destroyedTargets')} | Mission: {self.latest_mission}")
            except Exception:
                pass

    async def _consume_video(self, track: rtc.Track):
        stream = rtc.VideoStream(track)
        async for event in stream:
            rgb = event.frame.convert(rtc.VideoBufferType.RGB24)
            self.latest_frame = np.frombuffer(rgb.data, dtype=np.uint8).reshape((rgb.height, rgb.width, 3)).copy()

    async def connect(self, url: str = LIVEKIT_URL):
        token = mint_controller_token()
        await self.room.connect(url, token)
        print(f"[LiveKit] Connected to {url} (room='{LIVEKIT_ROOM}') as '{CONTROLLER_IDENTITY}'")

        # Bind existing remote video tracks if robot is already publishing
        for participant in self.room.remote_participants.values():
            for pub in participant.track_publications.values():
                if pub.track and pub.track.kind == rtc.TrackKind.KIND_VIDEO:
                    self._video_task = asyncio.create_task(self._consume_video(pub.track))

    async def send_teleop_axes(self, throttle: float = 0.0, pitch: float = 0.0, roll: float = 0.0, yaw: float = 0.0):
        """Send continuous low-latency flight teleoperation axes (-1.0 to +1.0) over LiveKit DataChannel."""
        payload = {
            "type": "teleop:control",
            "axes": {
                "throttle": float(np.clip(throttle, -1.0, 1.0)),
                "pitch": float(np.clip(pitch, -1.0, 1.0)),
                "roll": float(np.clip(roll, -1.0, 1.0)),
                "yaw": float(np.clip(yaw, -1.0, 1.0)),
            },
            "timestamp": int(time.time() * 1000),
        }
        await self.room.local_participant.publish_data(
            json.dumps(payload).encode("utf-8"), reliable=False, topic="teleop"
        )

    async def call_rpc(self, method: str, params: dict | None = None) -> dict:
        """Invoke a remote procedure call on the robot via LiveKit RPC."""
        payload_str = json.dumps(params or {})
        response_str = await self.room.local_participant.perform_rpc(
            destination_identity=ROBOT_IDENTITY,
            method=method,
            payload=payload_str,
        )
        try:
            return json.loads(response_str)
        except Exception:
            return {"response": response_str}

    async def publish_detections(self, detections: list[dict], width: int = 640, height: int = 360):
        """Send AI bounding-box annotations to display on the drone pilot's HUD."""
        payload = {
            "type": "ai:vision_annotations",
            "detections": detections,
            "width": width,
            "height": height,
            "timestamp": int(time.time() * 1000),
        }
        await self.room.local_participant.publish_data(
            json.dumps(payload).encode("utf-8"), reliable=True, topic="vision"
        )

    async def disconnect(self):
        if self._video_task:
            self._video_task.cancel()
        await self.room.disconnect()

drone = LiveKitDroneRobotClient()
await drone.connect()

## 2. Test LiveKit Robotics RPC Methods
Executes remote actions on the simulated drone over the LiveKit RPC channel.

In [ ]:
# Query current drone & mission state
state = await drone.call_rpc("teleop.get_state")
print("Drone State:", json.dumps(state, indent=2))

# Set speed mode to SPORT
res = await drone.call_rpc("teleop.set_speed_mode", {"mode": "SPORT"})
print("Speed Mode Response:", res)

# Arm and takeoff to 15m
takeoff_res = await drone.call_rpc("teleop.takeoff", {"altitude": 15})
print("Takeoff Response:", takeoff_res)

## 3. Real-Time Video Stream & Teleoperation Loop
Displays the live WebRTC camera stream inside Jupyter and sends periodic forward dive inputs.

In [ ]:
async def stream_and_teleop(duration_sec: float = 10.0, fps: float = 12.0):
    interval = 1.0 / fps
    t_end = time.time() + duration_sec

    while time.time() < t_end:
        # Send gentle forward pitch input (+0.30 forward, +0.10 climb)
        await drone.send_teleop_axes(throttle=0.10, pitch=0.30, roll=0.0, yaw=0.0)

        if drone.latest_frame is not None:
            img = Image.fromarray(drone.latest_frame).resize((480, 270))
            buf = io.BytesIO()
            img.save(buf, format="JPEG", quality=85)
            clear_output(wait=True)
            display(IPyImage(data=buf.getvalue()))

        if drone.latest_mission:
            print(f"🎯 Targets Hit: {drone.latest_mission.get('targetsHit')}/{drone.latest_mission.get('totalTargets')} | Drones Left: {drone.latest_mission.get('dronesRemaining')}")

        await asyncio.sleep(interval)

    # Neutralize controls at end of test
    await drone.send_teleop_axes(0, 0, 0, 0)
    print("Teleoperation test completed.")

await stream_and_teleop(duration_sec=5.0)

## 4. Collect Kaggle Dataset (Images + Telemetry + Mission State)
Collects synchronized frames and telemetry directly from WebRTC into `./dataset` for offline policy or vision training on **Kaggle**.

In [ ]:
async def collect_kaggle_dataset(num_samples: int = 50, interval_sec: float = 0.25, out_dir: str = "./dataset"):
    out_path = Path(out_dir)
    frames_dir = out_path / "images"
    frames_dir.mkdir(parents=True, exist_ok=True)
    records = []

    print(f"Collecting {num_samples} samples into {out_path.resolve()}...")
    for idx in range(num_samples):
        if drone.latest_frame is not None:
            img_name = f"frame_{idx:05d}.jpg"
            Image.fromarray(drone.latest_frame).save(frames_dir / img_name, quality=92)
            records.append({
                "image": img_name,
                "timestamp": time.time(),
                "telemetry": drone.latest_telemetry,
                "mission": drone.latest_mission,
            })
        await asyncio.sleep(interval_sec)

    with open(out_path / "metadata.jsonl", "w", encoding="utf-8") as f:
        for rec in records:
            f.write(json.dumps(rec) + "\n")

    print(f"✅ Successfully collected {len(records)} Kaggle samples!")

await collect_kaggle_dataset(num_samples=20)